# v15 — does restoring the scenario descriptors change the LSTM verdict?

## The one deviation worth testing

An audit against fifteen published N-CMAPSS studies put our pipeline inside normal practice on every
protocol dimension except one: **we exclude W, the scenario descriptors** (altitude, Mach number,
throttle resolver angle, fan-inlet temperature). Eleven of the fifteen include them. We drop them
because the reference method does.

That deviation is the one most likely to disadvantage a recurrent branch, for a reason measured
directly in v14: **about 84% of sensor variance on DS05 lies within flights rather than between
them**, and not one of the 28 channels is dominated by between-flight variance. That within-flight
variation is flight condition — which is exactly what W encodes.

With W supplied, a network can attribute sensor movement to operating regime and isolate the
degradation signal. Without it, the same movement arrives as unexplained nuisance in the sensor
channels and the network must infer flight state from the sensors before it can do anything else. A
recurrent branch asked to integrate a sequence dominated by unexplained regime variation has a
materially harder task than one handed the regime directly.

So the claim "the recurrent branch does not earn its place" may be a property of the architecture, or
it may be a property of withholding W. This notebook separates them.

## Design

A 2 × 3 factorial, paired on seed throughout:

| | full hybrid | CNN only | LSTM only |
|---|---|---|---|
| **28 channels** (X_s + X_v, as used so far) | — | — | — |
| **32 channels** (W + X_s + X_v, standard practice) | — | — | — |

Six cells at 30 seeds is 180 runs, roughly 2 hours. The seeds are the same ones used in v13, so the
28-channel column should reproduce v13 and acts as a built-in control: if it does not, something
other than W has changed and the comparison is void.

## Pre-registration

Fixed before running, not to be revised afterwards.

| | |
|---|---|
| seeds | 30, paired across every cell |
| alpha | 0.05 two-sided |
| equivalence margin | 0.40 RMSE, capped-55, as in v13 |
| **primary endpoint** | the **interaction**: does the (LSTM-only − hybrid) difference change between 28 and 32 channels? |
| secondary | main effect of W; branch effects within each feature set |

The primary endpoint is deliberately the interaction rather than the main effect. Whether W improves
accuracy overall is mildly interesting; whether it changes the verdict on the recurrent branch is the
question the paper turns on.

## Both outcomes are publishable

- **The LSTM stays unhelpful with W present.** The finding then holds under standard input
  conventions, is corroborated independently by CruiseBench (where TSMixer, having no recurrence,
  wins a fixed-protocol benchmark against LSTM and GRU baselines), and can be stated without
  qualification.
- **The LSTM becomes useful with W present.** The finding becomes sharper: the recurrent branch is
  worth its cost only when the operating regime is supplied explicitly, because otherwise it spends
  its capacity inferring flight state instead of degradation. That is a better result than the one we
  currently have.

In [ ]:
import os, json, gc, time, random, warnings
os.environ['PYTHONHASHSEED']='42'; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.metrics import mean_squared_error
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, Dropout, Dense, LSTM, Bidirectional,
                                     Concatenate, GlobalAveragePooling1D, Rescaling)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)

# ---------------- PRE-REGISTERED ----------------
SEED=42; N_SEEDS=30; ALPHA=0.05; MARGIN=0.40
SEEDS=[SEED+1000+i for i in range(N_SEEDS)]     # identical to v13
PRIMARY='interaction: (LSTM only - hybrid) at 32ch vs at 28ch'
# ------------------------------------------------
SUB=50; VAL_UNITS=[5,6]; CAP=55; EPOCHS=100; PATIENCE=15; T=20
print('TF',tf.__version__,'| GPU',bool(tf.config.list_physical_devices('GPU')))
print('%d seeds (same as v13), margin %.2f'%(N_SEEDS,MARGIN))

## Data — the two feature sets

`X_s` are the fourteen physical sensors, `X_v` the fourteen virtual sensors, `W` the four scenario
descriptors. Scaler statistics are computed separately for each feature set, from training rows only.

In [ ]:
def read_sub(d,step,chunk=400_000):
    n=d.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=d[a:b]; f=(-a)%step
        if f<len(blk): out.append(blk[f::step].astype(np.float32,copy=False))
    return np.concatenate(out)

with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=read_sub(h['W_dev'],SUB);  Sd=read_sub(h['X_s_dev'],SUB);  Vd=read_sub(h['X_v_dev'],SUB)
    Ad=read_sub(h['A_dev'],SUB);  Yd=read_sub(h['Y_dev'],SUB)
    Wt=read_sub(h['W_test'],SUB); St=read_sub(h['X_s_test'],SUB); Vt=read_sub(h['X_v_test'],SUB)
    At=read_sub(h['A_test'],SUB); Yt=read_sub(h['Y_test'],SUB)
UD,CD,YD=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
UT,CT,YT=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
FEAT={'28ch':(np.hstack([Sd,Vd]),       np.hstack([St,Vt])),
      '32ch':(np.hstack([Wd,Sd,Vd]),    np.hstack([Wt,St,Vt]))}
print('W provides %d channels; X_s %d; X_v %d'%(Wd.shape[1],Sd.shape[1],Vd.shape[1]))
del Wd,Sd,Vd,Ad,Yd,Wt,St,Vt,At,Yt; gc.collect()
TR_M=~np.isin(UD,VAL_UNITS); VA_M=np.isin(UD,VAL_UNITS)

def windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]; bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

DATA={}
for name,(FD,FT) in FEAT.items():
    tr=FD[TR_M]; mn,mx=tr.min(0),tr.max(0); rg=np.where((mx-mn)>1e-8,mx-mn,1.0)
    Xtr,ytr=windows(FD,UD,CD,YD,TR_M,T)
    Xva,yva=windows(FD,UD,CD,YD,VA_M,T)
    Xte,yte=windows(FT,UT,CT,YT,np.ones(len(YT),bool),T)
    DATA[name]=dict(Xtr=Xtr,ytr=np.minimum(ytr,CAP),Xva=Xva,yva=np.minimum(yva,CAP),
                    Xte=Xte,yte=np.minimum(yte,CAP),nfeat=FD.shape[1],
                    scale=(1.0/rg).astype(np.float32),offset=(-mn/rg).astype(np.float32))
    triv=float(np.sqrt(mean_squared_error(DATA[name]['yte'],
          np.full(len(DATA[name]['yte']),DATA[name]['ytr'].mean()))))
    DATA[name]['triv']=triv
    print('%s: %d channels | train %s | test %s | trivial %.3f | %.0f MB'
          %(name,FD.shape[1],format(len(Xtr),','),format(len(Xte),','),triv,
            (Xtr.nbytes+Xva.nbytes+Xte.nbytes)/1e6))
    del tr; gc.collect()
del FEAT; gc.collect()
assert abs(DATA['28ch']['triv']-DATA['32ch']['triv'])<1e-6, 'targets differ between feature sets'
print('target identical across feature sets - only the inputs change')

In [ ]:
FINAL=dict(T=T,n_conv=2,f1=171,f_growth=1.021014565802962,kernel=7,
           lstm_units=57,n_lstm=1,bidir=0,fuse_filters=49,
           dropout_conv=0.20536647015555695,dropout_dense=0.36568564920359575,
           l2reg=4.080309957019978e-06,dense1=45,activation='relu',
           lr=0.0005334569884698839,huber_delta=16.334471957530912,batch=256,
           use_lstm=1,use_cnn=1)
VARIANTS=[('full hybrid',dict(FINAL)),
          ('CNN only',   dict(FINAL,use_lstm=0)),
          ('LSTM only',  dict(FINAL,use_cnn=0))]

def build(kw,seed,d):
    act=kw['activation']
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    kk=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),kernel_regularizer=reg)
    inp=Input(shape=(kw['T'],d['nfeat']))
    x=Rescaling(scale=d['scale'],offset=d['offset'])(inp)
    br=[]
    if kw['use_cnn']:
        c=x
        for i in range(kw['n_conv']):
            f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
            c=Conv1D(f,kw['kernel'],padding='same',activation=act,**kk)(c)
            if kw['dropout_conv']>0: c=Dropout(kw['dropout_conv'],seed=seed)(c)
        br.append(c)
    if kw['use_lstm']:
        r=x
        for j in range(kw['n_lstm']):
            cell=LSTM(kw['lstm_units'],return_sequences=True,**kk)
            r=Bidirectional(cell)(r) if kw['bidir'] else cell(r)
        br.append(r)
    if not br: raise ValueError('no branch enabled')
    h=br[0] if len(br)==1 else Concatenate(axis=-1)(br)
    h=Conv1D(kw['fuse_filters'],3,padding='same',activation=act,**kk)(h)
    h=GlobalAveragePooling1D()(h)
    if kw['dense1']>=8:
        h=Dense(kw['dense1'],activation=act,**kk)(h)
        if kw['dropout_dense']>0: h=Dropout(kw['dropout_dense'],seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),
              loss=tf.keras.losses.Huber(delta=kw['huber_delta'])); return m

ROWS=[]
def run(kw,seed,fs):
    d=DATA[fs]
    K.clear_session(); gc.collect()
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    m=build(kw,seed,d)
    m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=EPOCHS,
          batch_size=kw['batch'],verbose=0,
          callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                     ReduceLROnPlateau('val_loss',factor=0.5,patience=7,min_lr=1e-6)])
    p=m.predict(d['Xte'],batch_size=8192,verbose=0).ravel()
    return float(np.sqrt(mean_squared_error(d['yte'],np.clip(p,0,CAP))))

t0=time.time(); _=run(dict(FINAL),SEED,'28ch')
print('warm-up %.0f s -> projected %.1f h for %d runs'
      %(time.time()-t0,(time.time()-t0)*6*N_SEEDS/3600,6*N_SEEDS))

## The 2 × 3 factorial

In [ ]:
t0=time.time()
for fs in ['28ch','32ch']:
    for name,kw in VARIANTS:
        for s in SEEDS:
            ROWS.append(dict(feat=fs,variant=name,seed=s,capped=run(kw,s,fs)))
        q=[r['capped'] for r in ROWS if r['feat']==fs and r['variant']==name]
        print('  %-6s %-13s %.4f +/- %.4f   (%.0f min)'
              %(fs,name,np.mean(q),np.std(q,ddof=1),(time.time()-t0)/60))
        pd.DataFrame(ROWS).to_csv('ncmapss_v15.csv',index=False)
D=pd.DataFrame(ROWS); print('done in %.1f h'%((time.time()-t0)/3600))

## Analysis

The control check comes first: the 28-channel hybrid must reproduce v13's 5.166 ± 0.304. If it does
not, W is not the only thing that changed and nothing below can be attributed to it.

In [ ]:
def tost(d,margin,alpha=ALPHA):
    n=len(d); se=d.std(ddof=1)/np.sqrt(n); df=n-1
    return max(stats.t.sf((d.mean()+margin)/se,df), stats.t.cdf((d.mean()-margin)/se,df))
def pick(fs,v): return D[(D.feat==fs)&(D.variant==v)].sort_values('seed').capped.values

h28=pick('28ch','full hybrid')
print('CONTROL: 28-channel hybrid %.4f +/- %.4f   (v13 reported 5.166 +/- 0.304)'%(h28.mean(),h28.std(ddof=1)))
print('         difference from v13: %+.4f  %s'%(h28.mean()-5.166,
      'consistent' if abs(h28.mean()-5.166)<0.15 else 'CHECK - the control did not reproduce'))
print()
print('='*94)
print('%-8s %-13s %9s %9s %10s %10s %10s'%('features','variant','mean','sd','vs hybrid','p','p TOST'))
print('='*94)
for fs in ['28ch','32ch']:
    base=pick(fs,'full hybrid')
    print('%-8s %-13s %9.4f %9.4f'%(fs,'full hybrid',base.mean(),base.std(ddof=1)))
    for v in ['CNN only','LSTM only']:
        q=pick(fs,v); d=q-base
        _,p=stats.ttest_rel(q,base); pe=tost(d,MARGIN)
        print('%-8s %-13s %9.4f %9.4f %+10.4f %10.5f %10.5f  %s'
              %(fs,v,q.mean(),q.std(ddof=1),d.mean(),p,pe,
                'different' if p<ALPHA else ('equivalent' if pe<ALPHA else 'inconclusive')))
print()
print('MAIN EFFECT OF W (32ch minus 28ch, paired on seed):')
for v in ['full hybrid','CNN only','LSTM only']:
    d=pick('32ch',v)-pick('28ch',v); _,p=stats.ttest_rel(pick('32ch',v),pick('28ch',v))
    print('  %-13s %+.4f   p = %.5f  %s'%(v,d.mean(),p,'W helps' if (d.mean()<0 and p<ALPHA)
          else ('W hurts' if (d.mean()>0 and p<ALPHA) else 'no effect')))
print()
print('PRIMARY ENDPOINT - interaction')
d28=pick('28ch','LSTM only')-pick('28ch','full hybrid')
d32=pick('32ch','LSTM only')-pick('32ch','full hybrid')
inter=d32-d28; t_,p_=stats.ttest_rel(d32,d28)
print('  (LSTM only - hybrid) at 28ch : %+.4f'%d28.mean())
print('  (LSTM only - hybrid) at 32ch : %+.4f'%d32.mean())
print('  interaction                  : %+.4f   t = %+.2f   p = %.5f'%(inter.mean(),t_,p_))
print()
if p_<ALPHA:
    print('  => W CHANGES the verdict on the recurrent branch. The earlier finding was')
    print('     specific to withholding the scenario descriptors, and must be restated.')
else:
    print('  => W does NOT change the verdict on the recurrent branch. The finding holds')
    print('     under standard input conventions and can be stated without that caveat.')
out=dict(n_seeds=N_SEEDS,margin=MARGIN,primary=PRIMARY,control_28ch=float(h28.mean()),
         interaction=float(inter.mean()),interaction_p=float(p_),
         cells={'%s|%s'%(f,v):[float(pick(f,v).mean()),float(pick(f,v).std(ddof=1))]
                for f in ['28ch','32ch'] for v in ['full hybrid','CNN only','LSTM only']})
json.dump(out,open('ncmapss_v15_summary.json','w'),indent=2)
D.to_csv('ncmapss_v15.csv',index=False)
print(); print('saved ncmapss_v15.csv and ncmapss_v15_summary.json')

## Figure

In [ ]:
plt.rcParams.update({'font.size':8,'font.family':'serif'})
fig,ax=plt.subplots(figsize=(4.4,2.7),dpi=300)
w=0.35; xs=np.arange(3); vs=['full hybrid','CNN only','LSTM only']
for k,(fs,col) in enumerate([('28ch','#888888'),('32ch','#0070C0')]):
    m=[pick(fs,v).mean() for v in vs]; e=[pick(fs,v).std(ddof=1) for v in vs]
    ax.bar(xs+(k-0.5)*w,m,w,yerr=e,capsize=3,color=col,
           label='%s%s'%(fs,' (W included)' if fs=='32ch' else ' (W excluded)'))
ax.set_xticks(xs); ax.set_xticklabels(vs)
ax.set_ylabel('Test RMSE (capped-55)')
lo=min(pick(f,v).mean()-pick(f,v).std(ddof=1) for f in ['28ch','32ch'] for v in vs)
hi=max(pick(f,v).mean()+pick(f,v).std(ddof=1) for f in ['28ch','32ch'] for v in vs)
ax.set_ylim(lo-0.2*(hi-lo),hi+0.25*(hi-lo))
ax.set_title('Do the scenario descriptors change the branch verdict? n = %d'%N_SEEDS,fontsize=8.5)
ax.legend(fontsize=6.5); ax.grid(axis='y',alpha=0.25)
plt.tight_layout(); plt.savefig('fig_v15_W.png',dpi=300,bbox_inches='tight')
print('wrote fig_v15_W.png'); plt.show()